In [1]:
import os

# You can only use one GPU at a time
os.environ["CUDA_VISIBLE_DEVICES"] = "6"
os.environ["OMP_NUM_THREADS"] = "16"
os.environ["MKL_NUM_THREADS"] = "16"

In [2]:
import torch

if torch.cuda.is_available():
    torch.cuda.set_device(0)

    print("CUDA is available.")
    print(f"Number of CUDA devices: {torch.cuda.device_count()}")
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")
else:
    print("CUDA is not available.")

CUDA is available.
Number of CUDA devices: 1
Current device: 0
Device name: NVIDIA A100-PCIE-40GB


In [3]:
# Load your model checkpoint
from safetensors.torch import load_file
model_ckpt = "/path/to/additional_evaluations/downloaded_models/models/sdxl/first_stage/version_0/checkpoints/latest-step=0/tmp.safetensors"

class_path = "vv.models.sgm_copy.sgm.models.diffusion.DiffusionEngine"
model_args = {
    "scale_factor": 0.13025,
    "disable_first_stage_autocast": True,
    "denoiser_config": {
        "target": "vv.models.sgm_copy.sgm.modules.diffusionmodules.denoiser.DiscreteDenoiser",
        "params": {
            "num_idx": 1000,
            "scaling_config": {
                "target": "vv.models.sgm_copy.sgm.modules.diffusionmodules.denoiser_scaling.EpsScaling",
            },
            "discretization_config": {
                "target": "vv.models.sgm_copy.sgm.modules.diffusionmodules.discretizer.LegacyDDPMDiscretization",
            },
        },
    },
    "network_config": {
        "target": "vv.models.sgm_copy.sgm.modules.diffusionmodules.openaimodel.UNetModel",
        "params": {
            "adm_in_channels": 2816,
            "num_classes": "sequential",
            "use_checkpoint": True,
            "in_channels": 4,
            "out_channels": 4,
            "model_channels": 320,
            "attention_resolutions": [4, 2],
            "num_res_blocks": 2,
            "channel_mult": [1, 2, 4],
            "num_head_channels": 64,
            "use_linear_in_transformer": True,
            "transformer_depth": [1, 2, 10],
            "context_dim": 2048,
            "spatial_transformer_attn_type": "softmax-xformers",
        },
    },
    "conditioner_config": {
        "target": "vv.models.sgm_copy.sgm.modules.GeneralConditioner",
        "params": {
            "emb_models": [
                {
                    "is_trainable": False,
                    "input_key": "txt",
                    "target": "vv.models.sgm_copy.sgm.modules.encoders.modules.FrozenCLIPEmbedder",
                    "params": {
                        "layer": "hidden",
                        "layer_idx": 11,
                    },
                },
                {
                    "is_trainable": False,
                    "input_key": "txt",
                    "target": "vv.models.sgm_copy.sgm.modules.encoders.modules.FrozenOpenCLIPEmbedder2",
                    "params": {
                        "arch": "ViT-bigG-14",
                        "version": "laion2b_s39b_b160k",
                        "freeze": True,
                        "layer": "penultimate",
                        "always_return_pooled": True,
                        "legacy": False,
                    },
                },
                {
                    "is_trainable": False,
                    "input_key": "original_size_as_tuple",
                    "target": "vv.models.sgm_copy.sgm.modules.encoders.modules.ConcatTimestepEmbedderND",
                    "params": {
                        "outdim": 256,
                    },
                },
                {
                    "is_trainable": False,
                    "input_key": "crop_coords_top_left",
                    "target": "vv.models.sgm_copy.sgm.modules.encoders.modules.ConcatTimestepEmbedderND",
                    "params": {
                        "outdim": 256,
                    },
                },
                {
                    "is_trainable": False,
                    "input_key": "target_size_as_tuple",
                    "target": "vv.models.sgm_copy.sgm.modules.encoders.modules.ConcatTimestepEmbedderND",
                    "params": {
                        "outdim": 256,
                    },
                },
            ],
        },
    },
    "first_stage_config": {
        "target": "vv.models.sgm_copy.sgm.models.autoencoder.AutoencoderKL",
        "params": {
            "embed_dim": 4,
            "monitor": "val/rec_loss",
            "ddconfig": {
                "attn_type": "vanilla-xformers",
                "double_z": True,
                "z_channels": 4,
                "resolution": 256,
                "in_channels": 3,
                "out_ch": 3,
                "ch": 128,
                "ch_mult": [1, 2, 4, 4],
                "num_res_blocks": 2,
                "attn_resolutions": [],
                "dropout": 0.0,
            },
            "lossconfig": {
                "target": "torch.nn.Identity",
            },
        },
    },
}

state_dict = load_file(model_ckpt)

module_path, class_name = class_path.rsplit(".", 1)
_module = __import__(module_path, fromlist=[class_name])
module = getattr(_module, class_name)
model = module(**model_args)

print("Loading model weights...")
model.load_state_dict(state_dict, strict=False)

# Move to GPU and set to eval mode
model = model.cuda().eval()

print("Model successfully loaded!")

./venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:vv.models.sgm_copy.sgm.modules.diffusionmodules.openaimodel:Building a Downsample layer with 2 dims.
INFO:vv.models.sgm_copy.sgm.modules.diffusionmodules.openaimodel:  --> settings are: 
 in-chn: 320, out-chn: 320, kernel-size: 3, stride: 2, padding: 1
INFO:vv.models.sgm_copy.sgm.modules.diffusionmodules.openaimodel:Building a Downsample layer with 2 dims.
INFO:vv.models.sgm_copy.sgm.modules.diffusionmodules.openaimodel:  --> settings are: 
 in-chn: 640, out-chn: 640, kernel-size: 3, stride: 2, padding: 1
INFO:root:Loaded ViT-bigG-14 model config.


Initialized embedder #0: FrozenCLIPEmbedder with 123060480 params. Trainable: False


INFO:root:Loading pretrained ViT-bigG-14 weights (laion2b_s39b_b160k).
INFO:vv.models.sgm_copy.sgm.modules.diffusionmodules.model:making attention of type 'vanilla-xformers' with 512 in_channels
INFO:vv.models.sgm_copy.sgm.modules.diffusionmodules.model:building MemoryEfficientAttnBlock with 512 in_channels...
INFO:vv.models.sgm_copy.sgm.modules.diffusionmodules.model:Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
INFO:vv.models.sgm_copy.sgm.modules.diffusionmodules.model:making attention of type 'vanilla-xformers' with 512 in_channels
INFO:vv.models.sgm_copy.sgm.modules.diffusionmodules.model:building MemoryEfficientAttnBlock with 512 in_channels...


Initialized embedder #1: FrozenOpenCLIPEmbedder2 with 694659841 params. Trainable: False
Initialized embedder #2: ConcatTimestepEmbedderND with 0 params. Trainable: False
Initialized embedder #3: ConcatTimestepEmbedderND with 0 params. Trainable: False
Initialized embedder #4: ConcatTimestepEmbedderND with 0 params. Trainable: False
Loading model weights...
Model successfully loaded!


In [4]:
first_stage_model = model.first_stage_model
print(f"First stage model: {first_stage_model}")

First stage model: AutoencoderKL(
  (encoder): Encoder(
    (conv_in): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (down): ModuleList(
      (0): Module(
        (block): ModuleList(
          (0-1): 2 x ResnetBlock(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 128, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          )
        )
        (attn): ModuleList()
        (downsample): Downsample(
          (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(2, 2))
        )
      )
      (1): Module(
        (block): ModuleList(
          (0): ResnetBlock(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (conv1): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 

In [5]:
import os
import torch
import lightning as L

first_stage_model = model.first_stage_model

if isinstance(first_stage_model, L.LightningModule):
    ckpt_dir = os.path.dirname(model_ckpt)
    first_stage_model_ckpt_path = os.path.join(ckpt_dir, "first_stage_model.ckpt")

    checkpoint = {
        "state_dict": first_stage_model.state_dict(),
        "hyper_parameters": getattr(first_stage_model, 'hparams', {}),
        "pytorch-lightning_version": L.__version__,
    }

    torch.save(checkpoint, first_stage_model_ckpt_path)
    print(f"First stage LightningModule checkpoint saved to {first_stage_model_ckpt_path}")
else:
    print("first_stage_model is not a LightningModule. Cannot save full checkpoint.")


First stage LightningModule checkpoint saved to /path/to/additional_evaluations/downloaded_models/models/sdxl/first_stage/version_0/checkpoints/latest-step=0/first_stage_model.ckpt


# Try to load the first stage directly

In [6]:
from vv.models.latent_diffusion_copy.KL_autoencoder import AutoencoderKL

model_ckpt = "/path/to/additional_evaluations/downloaded_models/models/sdxl/first_stage/version_0/checkpoints/latest-step=0/first_stage_model.ckpt"

model_args = {
    "learning_rate": 4.5e-06,
    "embed_dim": 4,
    "ddconfig": {
        "double_z": True,
        "z_channels": 4,
        "resolution": 256,
        "in_channels": 3,
        "out_ch": 3,
        "ch": 128,
        "ch_mult": [1, 2, 4, 4],
        "num_res_blocks": 2,
        "attn_resolutions": [],
        "dropout": 0.0,
    },
    "lossconfig": {
        "target": "vv.models.latent_diffusion_copy.dummy_loss.DummyLoss",
    },
}

print("Loading model...")

model = AutoencoderKL.load_from_checkpoint(
    model_ckpt,
    strict=False,
    **model_args
)

model = model.cuda().eval()

print("Model successfully loaded!")

Loading model...
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 4, 32, 32) = 4096 dimensions.
making attention of type 'vanilla' with 512 in_channels
Model successfully loaded!
